# ChatBeauty 실험 B — S1/S2 flat dense retrieval (Colab GPU)

1. 런타임 → 런타임 유형 변경 → **T4 GPU**
2. 로컬에서 `SPLIT=test python experiments/B/make_colab_bundle.py`로 번들을 만들고 `split -b 9m chatbeauty_B_bundle_test.zip chatbeauty_B_bundle_test.zip.part`로 나눈다
3. 왼쪽 파일 패널(또는 첫 셀의 업로드 창)로 조각 5개를 `/content`에 올린다
4. 모든 셀을 실행한다 (모델당 약 16분)
5. 마지막 셀의 `chatbeauty_B_outputs_{SPLIT}.zip`을 받아 로컬 `experiments/B/cache/`에 푼다

In [ ]:
SPLIT = 'test'  # 'valid' or 'test'
from google.colab import files
import glob, os
os.makedirs('/content/cb', exist_ok=True)
os.chdir('/content')
N_PARTS = 5
while len(glob.glob(f'/content/chatbeauty_B_bundle_{SPLIT}.zip.part*')) < N_PARTS:
    files.upload()
!cat /content/chatbeauty_B_bundle_{SPLIT}.zip.part* > /content/bundle.zip && cd /content/cb && unzip -oq /content/bundle.zip && mkdir -p experiments/B/cache && ls experiments/B/cache
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip -q install faiss-cpu sentence-transformers tqdm

In [ ]:
from huggingface_hub import snapshot_download
root = snapshot_download('kevin7548/chatbeauty-models', allow_patterns=['retrieval/bge-m3-finetuned-20260202-120852/*'])
FT = f'{root}/retrieval/bge-m3-finetuned-20260202-120852'
print(FT)

In [ ]:
# S1: fine-tuned (valid 기준값 0.334)
!cd /content/cb/experiments/B && SPLIT={SPLIT} python dense_retrieve.py SOURCE=S1 MODEL_PATH={FT} 2>&1 | grep -v 'it/s'

In [ ]:
# S2: zero-shot 원본 가중치 (valid 0.176)
!cd /content/cb/experiments/B && SPLIT={SPLIT} python dense_retrieve.py SOURCE=S2 MODEL_PATH=BAAI/bge-m3 2>&1 | grep -v 'it/s'

In [ ]:
!cd /content/cb/experiments/B/cache && zip -q /content/chatbeauty_B_outputs_{SPLIT}.zip *_{SPLIT}_S1.* *_{SPLIT}_S2.* && ls -la /content/chatbeauty_B_outputs_{SPLIT}.zip
# files.download(f'/content/chatbeauty_B_outputs_{SPLIT}.zip')  # 받을 때만 실행